# Lab 4: Parentage
## Please meet in Computer Lab FSH 136

### Goals
1. To calculate and interpret descriptive statistics (allele frequencies, number of alleles and heterozygosities) at several microsatellite loci for turkey (Meleagris gallopavo) from empirical population data
2. To quantify the relative benefits of displaying in coalitions or alone for turkey males. This will involve several steps:
    * assign turkey chicks to their fathers
    * estimate reproductive success of each male
    * estimate relatedness between turkey parents 
    * estimate inclusive fitness from individual reproductive success and thereproductive success of relatives
3. To test the effect of body length on reproductive success in steelhead. You will go through this exercise on your own (but best in the lab so we can help you), so there will be no written instructions

### The Procedure

Work in pairs – designate a recorder and a computer operator (we strongly encourage individualsto switch tasks throughout the lab), but make sure each person equally participates in the lab. All pairs will go through all exercises, and they should do so within the lab. Some exercises(especially paternity assignment of offspring) will be split between groups, so make sure to coordinate. Feel free to discuss questions within and among teams, and of course withinstructors. At the end of the lab, each pair will submit their report, which will be graded for bothgroup members. Remember to include both your names on the report. Nevertheless, everybody should take notes, because this stuff will be on the exam.

### Background

If we want to assign parentage then the number of loci, number of alleles and heterozygosity are important parameters predicting the likely success of parental assignment. For example, if wehave only one locus, and there is little variability at that locus, we won’t be able to exclude any adults as parents. We therefore need to calculate these parameters. First a few definitions:

#### What is a locus?
A locus is simply a physical location on a chromosome. A locus can be a single nucleotide, an entire gene, or a region of noncoding DNA. Loci (plural of locus) used in molecular ecology are specifically chosen for certain characteristics, such as universal amplification across several taxa(e.g. mtDNA cytochrome b) or high variability (e.g. microsatellites).

#### What is an allele?
An allele is a variant at a particular locus. A typical diploid individual can have only two alleles at each locus: one inherited from the mother, and one from the father. However, there can be dozens or even hundreds of different alleles present in a population. Confusingly, alleles can refer to different alleles (for example, a long and a short microsatellite allele) or simply copies of the locus even when they are identical (e.g. one diploid individual has two alleles). For microsatellite markers, we identify alleles by their size (in base pairs), which is determined by gel electrophoresis or automated capillary electrophoresis.

#### How do we obtain microsatellite genotypes? 
Microsatellites, also known as VNTRs (variable number of tandem repeats), are tandemly repeated regions of the DNA, for example, CACACACACA. The repeat unit can be between 2 and 6 basepairs (bp) long – in the above example, the repeat unit is two basepairs (i.e. adinucleotide microsatellite). The variability of microsatellites consists of the number of repeat units, in this case, five CA units. This can determined by designing primers in the flanking region of the microsatellite and estimating the number of repeat units from the size of the fragment. Fragment size is estimated on an electrophoresis gel. By comparison with a knownsize standard, we can estimate the size of each allele, which is the way they are commonlyidentified. For example, individual 1 may have a genotype of 150/152, which would mean that one allele is 150 bp long and the other 152 bp. There is a difference of two basepairs between the two alleles, which suggests that we are dealing with a dinucleotide microsatellite locus. 

![Image of microsatellite array](assets/microsat.jpg)

**Figure 1: Polymorphism of the trinucleotide microsatellite using gel electrophoresis.** Individuals are in vertical lanes (individuals 1-15 are labeled), and bands are microsatellite alleles. The first lane contains a 10 bp marker – the 180 bp and the 190 bp bands are visible and labeled. Numbers below each lane are the genotype of each individual in basepairs. (From: http://www.biomedcentral.com/1471-2164/9/287/figure/F3?highres=y)

### How are allele frequencies calculated?
Allele frequencies are calculated by dividing the number of a specific allele by the total number of alleles at each locus. For example, at a locus with three alleles (A,B,C) the frequency of allele A would be:

$$
p_{A}=\frac{2N_{AA}+N_{AB}+N_{AC}}{2N}
$$

where:

$p_{A}$: frequency of allele A

$N_{AA}$: number of AA homozygotes

$N_{AB}$: number of AB heterozygotes

$N_{AC}$: number of AC heterozygotes

$N$: sample size

Frequencies for alles B and C are calculated in a similar fashion.


### What is heterozygosity, and how is it calculated?
Observed heterozygosity (H) is a measure of genetic variation in a population; it tells us what proportion of individuals in the population are heterozygotes. It is calculated by dividing the number of heterozygous individuals by the sample size. In our example of a locus with three alleles (A,B,C), the formula for heterozygosity is:

$$
H=\frac{N_{AB}+N_{AC}+N_{BC}}{N}
$$

where:

$H$: heterozygosity

$N_{AB}$: number of AB heterozygotes

$N_{AC}$: number of AC heterozygotes

$N_{BC}$: number of BC heterozygotes

$N$: sample size

To calculate the average heterozygosity per locus, we simply calculate the heterozygosity for each locus and take the arithmetic mean over all loci.


In [ ]:
install.packages("tidyverse")


In [ ]:
library(tidyverse)

allele_frequencies <- function(file) {
  
  # Read the data
  dat <- read_csv(file)
  
  # Find the "a" copy of each locus
  allele_a <- names(dat)[str_ends(names(dat), "a")]
  
  # Calculate allele frequencies for each locus
  results <- map_dfr(allele_a, function(col_a) {
    
    # Find the matching "b" column
    col_b <- str_replace(col_a, "a$", "b")
    
    # Get the locus name
    locus <- str_remove(col_a, "a$")
    
    dat %>%
      select(all_of(c(col_a, col_b))) %>%
      
      # Put both allele copies into one column
      pivot_longer(
        cols = everything(),
        values_to = "allele"
      ) %>%
      
      # Remove missing alleles
      filter(!is.na(allele)) %>%
      
      # Count each allele
      count(allele) %>%
      
      # Calculate allele frequency
      mutate(
        frequency = n / sum(n),
        locus = locus
      ) %>%
      
      select(locus, allele, n, frequency)
  })
  
  return(results)
}